In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, IntegerType
)
from delta.tables import DeltaTable
from datetime import datetime


In [0]:


schema = StructType([
    StructField("S_No", LongType(), True),
    StructField("DataMart", StringType(), True),
    StructField("Target_Catalog", StringType(), True),
    StructField("Target_Schema", StringType(), True),
    StructField("Target_Table_Name", StringType(), True),
    StructField("Source", StringType(), True),
    StructField("Source_Path", StringType(), True),
    StructField("Source_File_Name", StringType(), True),
    StructField("Source_File_Format", StringType(), True),
    StructField("Primary_Fields", StringType(), True),
    StructField("Date_Field", StringType(), True),
    StructField("DataMart", StringType(), True),
    StructField("File_Type", StringType(), True),
    StructField("File_Count", IntegerType(), True),
    StructField("Is_Active", LongType(), True)
])


In [0]:
data = [
    (
        1,
        "healthcare_test",
        "iics",
        "casemix",
        "healthcare_test",
        "s3",
        "s3://cittabase-s3-test/Databricks-poc/iics_test/",
        "case_data_003.csv",
        "csv",
        "dta",
        10000,
        1
    ),
    (
        2,
        "healthcare_test",
        "iics",
        "casemix",
        "healthcare_test",
        "s3",
        "s3://cittabase-s3-test/Databricks-poc/iics_test/",
        "case_data_004.csv",
        "csv",
        "dta",
        10000,
        1
    )
]

In [0]:
df_control_table_creation = spark.createDataFrame(data, schema)

In [0]:
display(df_control_table_creation)

In [0]:
df_control_table_creation.write.format("delta").mode("overwrite").saveAsTable("iics.casemix.csv_file_control_test")

In [0]:
%sql
SELECT * FROM iics.casemix.csv_file_control_v1 

In [0]:
%sql
SELECT * FROM iics.casemix.csv_file_control_test -- WHERE S_No=2255;

In [0]:
%sql
drop table if exists iics.casemix.csv_file_logs_test;

In [0]:
%sql
select * from iics.casemix.csv_file_control_test

In [0]:
df_source = (spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv("s3://cittabase-s3-test/Databricks-poc/iics_test/case_data_005.csv")
    )

In [0]:
df_source = df_source.withColumn("loaded_at", F.lit(datetime.now()))

In [0]:
df_source.count()

In [0]:
display(df_source)

In [0]:
%sql
-- before merge
SELECT * FROM iics.casemix.healthcare_test;

In [0]:
target_table = DeltaTable.forName(spark, "iics.casemix.healthcare_test")


In [0]:
target_table.alias("tgt") \
    .merge(
        df_source.alias("src"),
        "tgt.CASE_SURR_ID = src.CASE_SURR_ID"
    ) \
    .whenMatchedUpdateAll(
        condition = "src.DH_UPDATED_DATE > tgt.DH_UPDATED_DATE",
    ) \
    .whenNotMatchedInsertAll() \
    .execute()

In [0]:
%sql
-- after merge
SELECT COUNT(*) FROM iics.casemix.healthcare_test;

In [0]:
%sql
-- before merge
select * from iics.casemix.healthcare_test --where CASE_SURR_ID IN ('10001','10002','10003','10004','10005','10006','10007','10008','10009','10010')

In [0]:
%sql
-- after merge
select * from iics.casemix.healthcare_test where CASE_SURR_ID IN ('10001','10002','10003','10004','10005','10006','10007','10008','10009','10010')

In [0]:
%sql
DROP TABLE IF EXISTS iics.casemix.healthcare_test